# 📊 Notebook Exploratoire — Détection de Tableaux PDF

> **Règle d'or** : Ce notebook ne modifie **aucun fichier du projet**.
> Il importe uniquement les fonctions existantes et explore les données.

**PDFs de test :**
- 🇨🇭 `UBS` — CH1522817787 (layout 2 colonnes, type BIL, EN)
- 🇫🇷 `Morgan Stanley` — BNP Decrement 4.60 (layout tabulaire coloré, FR)

---

## 1. Configuration & Imports

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict
import warnings
import pdfplumber
warnings.filterwarnings('ignore')

# ── Couleurs pour la visu ──────────────────────────────────────────
COLORS = {
    'word':    '#4C72B0',
    'gap':     '#DD8452',
    'col':     '#55A868',
    'line':    '#C44E52',
    'table':   '#8172B2',
    'header':  '#937860',
}

print("✅ Imports OK")

In [ ]:
# ── Import du projet (lecture seule) ──────────────────────────────
PROJECT_ROOT = Path(".")   # ← adapter si besoin
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from coord_extractor import extract_from_pdf
    PROJECT_AVAILABLE = True
    print("✅ coord_extractor importé depuis le projet")
except ImportError:
    PROJECT_AVAILABLE = False
    print("⚠️  coord_extractor non trouvé — fallback pdfplumber direct (paramètres identiques au projet)")

---
## 2. Fonctions d'extraction (fallback identique au projet)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  Dataclasses miroir du projet
# ══════════════════════════════════════════════════════════════════
@dataclass
class Word:
    text:   str
    x0:     float
    top:    float
    x1:     float
    bottom: float
    page:   int

@dataclass
class Line:
    words: List[Word]
    y:     float
    page:  int

    @property
    def text(self):  return " ".join(w.text for w in self.words)
    @property
    def x_positions(self): return [w.x0 for w in self.words]
    @property
    def x_gaps(self):
        xs = sorted(self.x_positions)
        return [xs[i+1] - xs[i] for i in range(len(xs)-1)]

# ══════════════════════════════════════════════════════════════════
#  Algorithme 1 — Extraction des mots (paramètres CRITIQUES du projet)
# ══════════════════════════════════════════════════════════════════
WORD_X_TOLERANCE = 1   # ⚠️  NE PAS MODIFIER — cf. rapport Section 2.1.3
WORD_Y_TOLERANCE = 1   # ⚠️  NE PAS MODIFIER

def extract_words(pdf_path: str) -> List[Word]:
    """Extraction identique au projet (x_tolerance=1, y_tolerance=1)."""
    if PROJECT_AVAILABLE:
        result = extract_from_pdf(pdf_path)
        return result.words
    words = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            for w in page.extract_words(
                x_tolerance=WORD_X_TOLERANCE,
                y_tolerance=WORD_Y_TOLERANCE,
                keep_blank_chars=False,
            ):
                words.append(Word(
                    text=w['text'], x0=w['x0'], top=w['top'],
                    x1=w['x1'], bottom=w['bottom'], page=i
                ))
    return words

# ══════════════════════════════════════════════════════════════════
#  Algorithme 2 — Regroupement en lignes (τy = 3.0 pt, cf. Section 2.2)
# ══════════════════════════════════════════════════════════════════
LINE_Y_THRESHOLD = 3.0  # calibré BIL — cf. Table 2 du rapport

def group_lines(words: List[Word], y_threshold: float = LINE_Y_THRESHOLD) -> List[Line]:
    """Algorithme 2 du projet."""
    if not words:
        return []
    sorted_words = sorted(words, key=lambda w: (w.page, w.top, w.x0))
    lines, current, y_cur, page_cur = [], [sorted_words[0]], sorted_words[0].top, sorted_words[0].page
    for w in sorted_words[1:]:
        if abs(w.top - y_cur) <= y_threshold and w.page == page_cur:
            current.append(w)
        else:
            current_s = sorted(current, key=lambda w: w.x0)
            lines.append(Line(words=current_s, y=np.mean([ww.top for ww in current_s]), page=page_cur))
            current, y_cur, page_cur = [w], w.top, w.page
    if current:
        current_s = sorted(current, key=lambda w: w.x0)
        lines.append(Line(words=current_s, y=np.mean([ww.top for ww in current_s]), page=page_cur))
    return lines

# ══════════════════════════════════════════════════════════════════
#  Algorithme 3 — Détection de colonnes par histogramme (Section 2.3)
# ══════════════════════════════════════════════════════════════════
COLUMN_GAP_MIN = 20.0  # ⚠️  cf. Section 5.1 du rapport

def detect_columns(lines: List[Line], page: int, bin_size: float = 5.0) -> List[float]:
    """Algorithme 3 du projet."""
    x_starts = [round(w.x0 / bin_size) * bin_size
                for l in lines if l.page == page
                for w in l.words]
    if not x_starts:
        return []
    counter = Counter(x_starts)
    columns = []
    for bin_x, count in counter.most_common():
        if count < 3:
            break
        if all(abs(bin_x - c) > COLUMN_GAP_MIN for c in columns):
            columns.append(bin_x)
    return sorted(columns)

# ══════════════════════════════════════════════════════════════════
#  Chargement complet d'un PDF
# ══════════════════════════════════════════════════════════════════
def load_pdf(path: str) -> dict:
    """Charge un PDF et retourne words, lines, columns par page."""
    words = extract_words(path)
    lines = group_lines(words)
    pages = sorted(set(w.page for w in words))
    columns = {p: detect_columns(lines, p) for p in pages}
    with pdfplumber.open(path) as pdf:
        n_pages = len(pdf.pages)
        page_w  = pdf.pages[0].width
        page_h  = pdf.pages[0].height
    return dict(words=words, lines=lines, columns=columns,
                n_pages=n_pages, page_w=page_w, page_h=page_h, path=path)

print("✅ Fonctions d'extraction définies")

---
## 3. Chargement des PDFs de test

In [ ]:
# ── Chemins des PDFs ──────────────────────────────────────────────
PDFS = {
    "UBS (CH1522817787)": "CH1522817787_TermsheetvomEmissionstag_en.pdf",
    "Morgan Stanley (BNP Decrement)": "ts-autocall-quotidien-bnp-decrement-460-septembre-2025-frip00001k13.pdf",
}

# ── Adapter ce préfixe à votre arborescence ───────────────────────
PDF_DIR = Path(".")   # ← dossier contenant les PDFs

# Chargement
data = {}
for name, fname in PDFS.items():
    path = str(PDF_DIR / fname)
    print(f"⏳ Chargement : {name}")
    doc = load_pdf(path)
    data[name] = doc
    print(f"   → {len(doc['words'])} mots | {len(doc['lines'])} lignes | {doc['n_pages']} pages")
    for p, cols in doc['columns'].items():
        if cols:
            frontier = (cols[0] + cols[1]) / 2 if len(cols) >= 2 else None
            print(f"   → Page {p+1} : colonnes détectées = {[round(c) for c in cols]}"
                  + (f" | frontière = {frontier:.0f} pt" if frontier else ""))
print("\n✅ Tous les PDFs chargés")

---
## 4. Visualisation Géométrique

> Scatter plot : chaque mot = un point. `x = x0`, `y = top` (inversé pour simuler la page).
> Activez/désactivez les options dans les paramètres.

In [ ]:
def plot_geometry(
    doc: dict,
    page: int = 0,
    title: str = "",
    # ── Options de debug ──────────────────────────────────────────
    show_text:    bool = True,   # afficher le texte sur chaque point
    show_x0:      bool = False,  # afficher la valeur x0 sous chaque point
    show_gaps:    bool = False,  # afficher les gaps inter-mots sur chaque ligne
    show_columns: bool = True,   # afficher les colonnes détectées (lignes verticales)
    show_lines:   bool = True,   # afficher les lignes détectées (lignes horizontales)
    figsize: tuple = (16, 20),
):
    words_p = [w for w in doc['words'] if w.page == page]
    lines_p = [l for l in doc['lines'] if l.page == page]
    cols    = doc['columns'].get(page, [])

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#FAFAFA')
    ax.invert_yaxis()
    ax.set_xlim(-5, doc['page_w'] + 5)
    ax.set_ylim(doc['page_h'] + 5, -5)
    ax.set_xlabel("x0 (pt)", fontsize=10)
    ax.set_ylabel("top (pt) — inversé", fontsize=10)
    ax.set_title(f"{title}  —  Page {page+1}", fontsize=13, fontweight='bold', pad=12)
    ax.grid(True, alpha=0.2, linestyle='--')

    # ── Lignes horizontales ───────────────────────────────────────
    if show_lines:
        for ln in lines_p:
            xs = [w.x0 for w in ln.words]
            ax.axhline(y=ln.y, xmin=0, xmax=1, color=COLORS['line'],
                       alpha=0.12, linewidth=0.8, linestyle='-')

    # ── Colonnes verticales ───────────────────────────────────────
    if show_columns and cols:
        for c in cols:
            ax.axvline(x=c, color=COLORS['col'], alpha=0.6, linewidth=1.5,
                       linestyle='--', label=f'Col x≈{c:.0f}')
        if len(cols) >= 2:
            frontier = (cols[0] + cols[1]) / 2
            ax.axvline(x=frontier, color='black', alpha=0.4, linewidth=1,
                       linestyle=':', label=f'Frontière x={frontier:.0f}')

    # ── Scatter des mots ──────────────────────────────────────────
    xs = [w.x0 for w in words_p]
    ys = [w.top for w in words_p]
    ax.scatter(xs, ys, s=18, color=COLORS['word'], alpha=0.7, zorder=5)

    # ── Texte ─────────────────────────────────────────────────────
    if show_text:
        for w in words_p:
            ax.text(w.x0, w.top - 1, w.text, fontsize=4.5, color='#222',
                    va='bottom', clip_on=True, zorder=6)

    # ── x0 numériques ─────────────────────────────────────────────
    if show_x0:
        for w in words_p:
            ax.text(w.x0, w.top + 3, f"{w.x0:.0f}", fontsize=3.5,
                    color=COLORS['gap'], va='top', clip_on=True, zorder=7)

    # ── Gaps ──────────────────────────────────────────────────────
    if show_gaps:
        for ln in lines_p:
            ws = sorted(ln.words, key=lambda w: w.x0)
            for i in range(len(ws) - 1):
                gap = ws[i+1].x0 - ws[i].x0
                mid_x = (ws[i].x0 + ws[i+1].x0) / 2
                ax.annotate(
                    f"{gap:.0f}",
                    xy=(mid_x, ln.y),
                    fontsize=3.5, color=COLORS['gap'],
                    ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.1', fc='#FFF3E0', ec='none', alpha=0.8),
                    zorder=8, clip_on=True,
                )

    # ── Légende ───────────────────────────────────────────────────
    handles = [
        mlines.Line2D([0],[0], marker='o', color='w', markerfacecolor=COLORS['word'],
                      markersize=6, label='Mot détecté'),
    ]
    if show_lines:
        handles.append(mlines.Line2D([0],[0], color=COLORS['line'], lw=1, label='Ligne détectée'))
    if show_columns and cols:
        handles.append(mlines.Line2D([0],[0], color=COLORS['col'], lw=1.5,
                                     linestyle='--', label='Colonne (histogramme)'))
    ax.legend(handles=handles, fontsize=7, loc='upper right')

    # ── Stats en bas ──────────────────────────────────────────────
    info = (f"Mots: {len(words_p)} | Lignes: {len(lines_p)} | "
            f"Colonnes: {[round(c) for c in cols]} | "
            f"Frontière: {(cols[0]+cols[1])/2:.0f} pt" if len(cols)>=2 else
            f"Mots: {len(words_p)} | Lignes: {len(lines_p)} | Colonnes: {[round(c) for c in cols]}")
    ax.text(0.01, 0.01, info, transform=ax.transAxes, fontsize=7,
            color='#555', va='bottom')

    plt.tight_layout()
    plt.show()

print("✅ Fonction plot_geometry définie")

In [ ]:
# ── 🔬 UBS — Page 1 (tableau Information on Underlying) ──────────
plot_geometry(
    data["UBS (CH1522817787)"],
    page=0,
    title="UBS CH1522817787",
    show_text=True,
    show_columns=True,
    show_lines=True,
    show_gaps=False,
    show_x0=False,
)

In [ ]:
# ── 🔬 Morgan Stanley — Page 1 (SOUS-JACENT + CARACTÉRISTIQUES) ──
plot_geometry(
    data["Morgan Stanley (BNP Decrement)"],
    page=0,
    title="Morgan Stanley — BNP Decrement",
    show_text=True,
    show_columns=True,
    show_lines=True,
    show_gaps=False,
    show_x0=False,
)

In [ ]:
# ── 🔬 Morgan Stanley — Page 2 (longue table autocall) ───────────
plot_geometry(
    data["Morgan Stanley (BNP Decrement)"],
    page=1,
    title="Morgan Stanley — Page 2 (table autocall)",
    show_text=True,
    show_columns=True,
    show_lines=True,
    show_gaps=True,    # ← gaps visibles sur cette page
    show_x0=False,
)

In [ ]:
# ── 🔬 UBS — Page 2 (Dates + Coupon tables) ──────────────────────
plot_geometry(
    data["UBS (CH1522817787)"],
    page=1,
    title="UBS — Page 2 (Dates + Coupon tables)",
    show_text=True,
    show_columns=True,
    show_lines=True,
    show_gaps=True,
    show_x0=False,
)

---
## 5. Algorithme de Détection de Tableaux (Version Exploratoire)

> Implémenté **uniquement dans ce notebook** — ne modifie pas le projet.

### Principe
Un tableau est détecté quand **plusieurs lignes consécutives** partagent des gaps `x[i+1] - x[i]` **réguliers** (faible variance) **sur au moins N_min lignes**.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ⚙️  PARAMÈTRES AJUSTABLES — modifier librement ici
# ══════════════════════════════════════════════════════════════════
PARAMS = dict(
    gap_tolerance     = 8.0,   # tolérance (pt) pour considérer deux gaps comme "égaux"
    min_lines         = 2,     # nombre minimum de lignes pour valider un tableau
    min_cols_in_table = 2,     # nombre minimum de colonnes dans un tableau
    min_words_per_line= 2,     # ignorer les lignes avec un seul mot
    max_y_gap         = 20.0,  # gap vertical max (pt) entre deux lignes du même tableau
    validation_thresh = 0.5,   # fraction minimum de lignes valides pour accepter le tableau
)

print("Paramètres actifs :")
for k, v in PARAMS.items():
    print(f"  {k:25s} = {v}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  Algorithme exploratoire — Détection par régularité des gaps
# ══════════════════════════════════════════════════════════════════

def compute_line_gaps(line: Line) -> List[float]:
    """Calcule les gaps entre les x0 des mots d'une ligne."""
    xs = sorted(w.x0 for w in line.words)
    return [xs[i+1] - xs[i] for i in range(len(xs)-1)]


def gaps_are_compatible(ref_gaps: List[float], line_gaps: List[float],
                         tol: float) -> bool:
    """Vérifie si deux listes de gaps sont suffisamment proches."""
    if len(ref_gaps) != len(line_gaps):
        return False
    return all(abs(g1 - g2) <= tol for g1, g2 in zip(ref_gaps, line_gaps))


@dataclass
class DetectedTable:
    lines:   List[Line]
    page:    int
    n_cols:  int
    col_xs:  List[float]   # positions x approximatives des colonnes
    ref_gaps:List[float]   # gaps de référence
    confidence: float      # fraction de lignes valides

    @property
    def n_rows(self): return len(self.lines)

    @property
    def y_start(self): return self.lines[0].y if self.lines else 0

    @property
    def y_end(self): return self.lines[-1].y if self.lines else 0


def detect_tables(lines: List[Line], page: int, params: dict) -> List[DetectedTable]:
    """
    Détecte les tableaux sur une page par régularité des gaps x.
    
    Logique :
    1. Filtrer les lignes avec assez de mots
    2. Pour chaque ligne candidate, tenter d'étendre un groupe
       en vérifiant la compatibilité des gaps avec la ligne de référence
    3. Si le groupe dépasse min_lines et min_cols → tableau détecté
    """
    tol      = params['gap_tolerance']
    min_l    = params['min_lines']
    min_c    = params['min_cols_in_table']
    min_w    = params['min_words_per_line']
    max_ygap = params['max_y_gap']
    val_thr  = params['validation_thresh']

    page_lines = sorted(
        [l for l in lines if l.page == page and len(l.words) >= min_w],
        key=lambda l: l.y
    )

    tables, used = [], set()

    for i, ref_line in enumerate(page_lines):
        if i in used:
            continue
        ref_gaps = compute_line_gaps(ref_line)
        if len(ref_gaps) < min_c - 1:
            continue

        group = [ref_line]
        valid_count = 1

        for j in range(i+1, len(page_lines)):
            next_line = page_lines[j]
            # Vérifier le gap vertical
            if next_line.y - group[-1].y > max_ygap:
                break
            line_gaps = compute_line_gaps(next_line)
            if gaps_are_compatible(ref_gaps, line_gaps, tol):
                group.append(next_line)
                used.add(j)
                valid_count += 1
            elif len(line_gaps) >= min_c - 1:
                # Ligne avec un nombre de colonnes différent → stop
                break
            else:
                # Ligne courte (ex: en-tête, continuation) → on inclut mais on n'update pas les gaps
                group.append(next_line)

        confidence = valid_count / len(group) if group else 0
        if len(group) >= min_l and valid_count >= min_l and confidence >= val_thr:
            col_xs = sorted(set(round(w.x0) for l in group for w in l.words))[:10]
            tables.append(DetectedTable(
                lines=group, page=page,
                n_cols=len(ref_gaps)+1,
                col_xs=col_xs,
                ref_gaps=ref_gaps,
                confidence=confidence,
            ))

    # Dédoublonnage : supprimer les tableaux qui se recoupent trop
    final = []
    for t in sorted(tables, key=lambda t: -len(t.lines)):
        overlap = any(
            len(set(id(l) for l in t.lines) & set(id(l) for l in f.lines)) > 1
            for f in final
        )
        if not overlap:
            final.append(t)
    return sorted(final, key=lambda t: t.y_start)


print("✅ Algorithme de détection défini")

---
## 6. Backtest sur les deux PDFs

In [ ]:
# ── Lancer la détection sur toutes les pages de tous les PDFs ─────
results = {}

for doc_name, doc in data.items():
    results[doc_name] = {}
    print(f"\n{'='*60}")
    print(f"📄 {doc_name}")
    print(f"{'='*60}")
    total_tables = 0
    for page in range(doc['n_pages']):
        tables = detect_tables(doc['lines'], page, PARAMS)
        results[doc_name][page] = tables
        if tables:
            total_tables += len(tables)
            print(f"\n  Page {page+1} → {len(tables)} tableau(x) détecté(s)")
            for k, t in enumerate(tables):
                print(f"    Tableau {k+1}: {t.n_rows} lignes × {t.n_cols} cols "
                      f"| y=[{t.y_start:.0f}→{t.y_end:.0f}] "
                      f"| confiance={t.confidence:.0%} "
                      f"| gaps={[round(g) for g in t.ref_gaps]}")
    print(f"\n  ✅ Total : {total_tables} tableaux sur {doc['n_pages']} pages")

---
## 7. Affichage des Tableaux en DataFrame

In [ ]:
def table_to_dataframe(table: DetectedTable, col_tol: float = 15.0) -> pd.DataFrame:
    """
    Convertit un DetectedTable en DataFrame pandas.
    Assigne chaque mot à une colonne par proximité de son x0 aux col_xs de référence.
    """
    col_refs = table.col_xs[:table.n_cols * 2]
    # Regrouper les col_refs en clusters
    clusters = []
    for x in sorted(col_refs):
        if not clusters or x - clusters[-1] > col_tol:
            clusters.append(x)
    if len(clusters) < 1:
        clusters = col_refs

    rows = []
    for line in table.lines:
        row = [''] * len(clusters)
        for w in line.words:
            # Trouver la colonne la plus proche
            dists = [abs(w.x0 - c) for c in clusters]
            col_idx = int(np.argmin(dists))
            if row[col_idx]:
                row[col_idx] += ' ' + w.text
            else:
                row[col_idx] = w.text
        rows.append(row)

    cols_labels = [f"Col_{i+1}" for i in range(len(clusters))]
    df = pd.DataFrame(rows, columns=cols_labels)
    # Nettoyer les colonnes entièrement vides
    df = df.loc[:, (df != '').any(axis=0)]
    return df


def display_all_tables(doc_name: str, results_doc: dict, max_tables: int = 20):
    """Affiche tous les tableaux détectés pour un PDF sous forme de DataFrames."""
    count = 0
    for page, tables in results_doc.items():
        for k, t in enumerate(tables):
            if count >= max_tables:
                print(f"... (affichage limité à {max_tables} tableaux)")
                return
            print(f"\n{'─'*70}")
            print(f"📋 {doc_name} | Page {page+1} | Tableau {k+1}")
            print(f"   {t.n_rows} lignes × {t.n_cols} colonnes "
                  f"| confiance={t.confidence:.0%} "
                  f"| gaps={[round(g) for g in t.ref_gaps]}")
            print(f"{'─'*70}")
            df = table_to_dataframe(t)
            display(df)
            count += 1

print("✅ Fonction table_to_dataframe définie")

In [ ]:
# ── UBS ──────────────────────────────────────────────────────────
display_all_tables("UBS (CH1522817787)", results["UBS (CH1522817787)"])

In [ ]:
# ── Morgan Stanley ────────────────────────────────────────────────
display_all_tables("Morgan Stanley (BNP Decrement)", results["Morgan Stanley (BNP Decrement)"])

---
## 8. Debug Visuel — Tableaux Surlignés sur la Géométrie

In [ ]:
def plot_tables_overlay(
    doc: dict,
    tables: List[DetectedTable],
    page: int,
    title: str = "",
    # ── Options debug ─────────────────────────────────────────────
    show_raw:     bool = True,   # mots non-tableau en gris
    show_cleaned: bool = True,   # mots tableau en couleur
    show_x0:      bool = False,
    show_gaps:    bool = False,
    figsize: tuple = (16, 20),
):
    words_p  = [w for w in doc['words'] if w.page == page]
    lines_p  = [l for l in doc['lines'] if l.page == page]
    cols_det = doc['columns'].get(page, [])

    # Mots appartenant à un tableau
    table_word_ids = set()
    for t in tables:
        for l in t.lines:
            for w in l.words:
                table_word_ids.add(id(w))

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#FAFAFA')
    ax.invert_yaxis()
    ax.set_xlim(-5, doc['page_w'] + 5)
    ax.set_ylim(doc['page_h'] + 5, -5)
    ax.set_xlabel("x0 (pt)", fontsize=10)
    ax.set_ylabel("top (pt) — inversé", fontsize=10)
    ax.set_title(f"{title} — Page {page+1} — Tableaux détectés",
                 fontsize=13, fontweight='bold', pad=12)
    ax.grid(True, alpha=0.15, linestyle='--')

    # ── Zones des tableaux ────────────────────────────────────────
    tab_colors = ['#E8F5E9','#E3F2FD','#FFF3E0','#F3E5F5','#FCE4EC']
    for k, t in enumerate(tables):
        if t.page != page:
            continue
        x_min = min(w.x0 for l in t.lines for w in l.words) - 3
        x_max = max(w.x1 for l in t.lines for w in l.words) + 3
        y_min = t.y_start - 4
        y_max = t.y_end + 4
        rect = mpatches.FancyBboxPatch(
            (x_min, y_min), x_max-x_min, y_max-y_min,
            boxstyle="round,pad=2",
            linewidth=1.5, edgecolor=COLORS['table'],
            facecolor=tab_colors[k % len(tab_colors)],
            alpha=0.55, zorder=2
        )
        ax.add_patch(rect)
        ax.text(x_min+2, y_min+1,
                f"T{k+1}: {t.n_rows}×{t.n_cols} ({t.confidence:.0%})",
                fontsize=6, color=COLORS['table'], fontweight='bold',
                va='top', zorder=9)

    # ── Mots hors tableau (gris) ──────────────────────────────────
    if show_raw:
        for w in words_p:
            if id(w) not in table_word_ids:
                ax.scatter(w.x0, w.top, s=12, color='#BBBBBB', alpha=0.5, zorder=4)
                ax.text(w.x0, w.top-1, w.text, fontsize=4, color='#999',
                        va='bottom', clip_on=True, zorder=5)

    # ── Mots dans un tableau (colorés) ───────────────────────────
    if show_cleaned:
        tc = ['#1565C0','#2E7D32','#E65100','#6A1B9A','#880E4F']
        for k, t in enumerate(tables):
            if t.page != page:
                continue
            c = tc[k % len(tc)]
            for l in t.lines:
                for w in l.words:
                    ax.scatter(w.x0, w.top, s=22, color=c, alpha=0.9, zorder=6)
                    ax.text(w.x0, w.top-1, w.text, fontsize=5, color=c,
                            va='bottom', fontweight='bold', clip_on=True, zorder=7)

    # ── Gaps si demandé ───────────────────────────────────────────
    if show_gaps:
        for t in tables:
            if t.page != page:
                continue
            for ln in t.lines:
                ws = sorted(ln.words, key=lambda w: w.x0)
                for i in range(len(ws)-1):
                    gap = ws[i+1].x0 - ws[i].x0
                    mid_x = (ws[i].x0 + ws[i+1].x0) / 2
                    ax.annotate(f"{gap:.0f}", xy=(mid_x, ln.y),
                                fontsize=4, color=COLORS['gap'], ha='center',
                                bbox=dict(boxstyle='round,pad=0.1', fc='#FFF9C4',
                                          ec='none', alpha=0.9),
                                zorder=10, clip_on=True)

    # ── x0 ────────────────────────────────────────────────────────
    if show_x0:
        for t in tables:
            if t.page != page:
                continue
            for ln in t.lines:
                for w in ln.words:
                    ax.text(w.x0, w.top+3, f"{w.x0:.0f}",
                            fontsize=3.5, color='#888', va='top', clip_on=True)

    plt.tight_layout()
    plt.show()

print("✅ Fonction plot_tables_overlay définie")

In [ ]:
# ── 🔬 UBS Page 1 — tableaux surlignés ───────────────────────────
plot_tables_overlay(
    data["UBS (CH1522817787)"],
    results["UBS (CH1522817787)"][0],
    page=0,
    title="UBS CH1522817787",
    show_raw=True,
    show_cleaned=True,
    show_gaps=True,
    show_x0=False,
)

In [ ]:
# ── 🔬 UBS Page 2 — Coupon Accrual + Early Redemption ────────────
plot_tables_overlay(
    data["UBS (CH1522817787)"],
    results["UBS (CH1522817787)"][1],
    page=1,
    title="UBS — Page 2",
    show_raw=True,
    show_cleaned=True,
    show_gaps=True,
)

In [ ]:
# ── 🔬 Morgan Stanley Page 1 ─────────────────────────────────────
plot_tables_overlay(
    data["Morgan Stanley (BNP Decrement)"],
    results["Morgan Stanley (BNP Decrement)"][0],
    page=0,
    title="Morgan Stanley — Page 1",
    show_raw=True,
    show_cleaned=True,
    show_gaps=True,
)

In [ ]:
# ── 🔬 Morgan Stanley Page 2 — longue table autocall ─────────────
plot_tables_overlay(
    data["Morgan Stanley (BNP Decrement)"],
    results["Morgan Stanley (BNP Decrement)"][1],
    page=1,
    title="Morgan Stanley — Page 2 (autocall)",
    show_raw=True,
    show_cleaned=True,
    show_gaps=False,
)

---
## 9. Résumé Global — Scorecard

In [ ]:
def build_scorecard(results: dict) -> pd.DataFrame:
    rows = []
    for doc_name, pages in results.items():
        for page, tables in pages.items():
            for k, t in enumerate(tables):
                rows.append(dict(
                    Document=doc_name,
                    Page=page+1,
                    Tableau=k+1,
                    Lignes=t.n_rows,
                    Colonnes=t.n_cols,
                    Y_debut=round(t.y_start),
                    Y_fin=round(t.y_end),
                    Confiance=f"{t.confidence:.0%}",
                    Gaps=[round(g) for g in t.ref_gaps],
                ))
    return pd.DataFrame(rows)

scorecard = build_scorecard(results)
print(f"\n📊 SCORECARD GLOBAL — {len(scorecard)} tableaux détectés\n")
display(scorecard)

In [ ]:
# ── Résumé par document ───────────────────────────────────────────
summary = (
    scorecard
    .groupby('Document')
    .agg(
        Total_tableaux=('Tableau','count'),
        Pages_avec_tableau=('Page', 'nunique'),
        Lignes_max=('Lignes','max'),
        Lignes_moy=('Lignes','mean'),
        Cols_max=('Colonnes','max'),
    )
    .round(1)
)
display(summary)

---
## 10. Analyse des Erreurs — Pourquoi un tableau est-il raté ?

> Cellule de debug : choisir un PDF et une page, puis inspecter ligne par ligne.

In [ ]:
# ── 🔧 Paramètres de debug ────────────────────────────────────────
DEBUG_DOC  = "UBS (CH1522817787)"      # ← changer ici
DEBUG_PAGE = 0                          # ← numéro de page (0-indexé)

doc   = data[DEBUG_DOC]
lines_p = sorted(
    [l for l in doc['lines'] if l.page == DEBUG_PAGE],
    key=lambda l: l.y
)

print(f"\n{'='*70}")
print(f"DEBUG — {DEBUG_DOC} | Page {DEBUG_PAGE+1}")
print(f"{'='*70}")
print(f"{'Y':>6} | {'#W':>3} | {'Gaps':^40} | Texte")
print('-'*100)

for ln in lines_p:
    gaps = compute_line_gaps(ln)
    gap_str = str([round(g) for g in gaps])[:38]
    text_preview = ln.text[:45]
    marker = '⬛' if len(ln.words) < PARAMS['min_words_per_line'] else ' '
    print(f"{ln.y:6.1f} | {len(ln.words):3d} | {gap_str:<40} | {marker} {text_preview}")

In [ ]:
# ── Même chose pour Morgan Stanley Page 2 (table autocall) ───────
DEBUG_DOC2  = "Morgan Stanley (BNP Decrement)"
DEBUG_PAGE2 = 1

doc2 = data[DEBUG_DOC2]
lines_p2 = sorted(
    [l for l in doc2['lines'] if l.page == DEBUG_PAGE2],
    key=lambda l: l.y
)

print(f"\n{'='*70}")
print(f"DEBUG — {DEBUG_DOC2} | Page {DEBUG_PAGE2+1}")
print(f"{'='*70}")
print(f"{'Y':>6} | {'#W':>3} | {'Gaps':^40} | Texte")
print('-'*100)

for ln in lines_p2:
    gaps = compute_line_gaps(ln)
    gap_str = str([round(g) for g in gaps])[:38]
    text_preview = ln.text[:45]
    marker = '⬛' if len(ln.words) < PARAMS['min_words_per_line'] else ' '
    print(f"{ln.y:6.1f} | {len(ln.words):3d} | {gap_str:<40} | {marker} {text_preview}")

---
## 11. Calibration Rapide — Impact des Paramètres

> Modifier `gap_tolerance` ou `min_lines` et relancer cette cellule pour observer l'effet immédiat.

In [ ]:
# ── Grid search sur gap_tolerance ────────────────────────────────
print("Impact de gap_tolerance sur le nombre de tableaux détectés\n")
print(f"{'gap_tol':>10} | {'UBS':>8} | {'Morgan Stanley':>16}")
print('-'*40)

for gt in [3, 5, 8, 12, 20, 30]:
    params_test = {**PARAMS, 'gap_tolerance': gt}
    counts = {}
    for doc_name, doc in data.items():
        total = sum(
            len(detect_tables(doc['lines'], p, params_test))
            for p in range(doc['n_pages'])
        )
        counts[doc_name] = total
    ubs = counts["UBS (CH1522817787)"]
    ms  = counts["Morgan Stanley (BNP Decrement)"]
    print(f"{gt:>10} | {ubs:>8} | {ms:>16}")

In [ ]:
# ── Impact de min_lines ───────────────────────────────────────────
print("Impact de min_lines sur le nombre de tableaux détectés\n")
print(f"{'min_lines':>10} | {'UBS':>8} | {'Morgan Stanley':>16}")
print('-'*40)

for ml in [2, 3, 4, 5, 6]:
    params_test = {**PARAMS, 'min_lines': ml}
    counts = {}
    for doc_name, doc in data.items():
        total = sum(
            len(detect_tables(doc['lines'], p, params_test))
            for p in range(doc['n_pages'])
        )
        counts[doc_name] = total
    ubs = counts["UBS (CH1522817787)"]
    ms  = counts["Morgan Stanley (BNP Decrement)"]
    print(f"{ml:>10} | {ubs:>8} | {ms:>16}")

---
## 12. Tester un Nouveau PDF

> Cellule clé-en-main. Remplacer `NEW_PDF_PATH` par le chemin d'un nouveau PDF à analyser.

In [ ]:
# ── 🆕 Ajouter un nouveau PDF ici ─────────────────────────────────
NEW_PDF_PATH = "votre_nouveau.pdf"   # ← modifier
NEW_PDF_NAME = "Nouveau PDF"

if Path(NEW_PDF_PATH).exists():
    print(f"⏳ Chargement : {NEW_PDF_NAME}")
    new_doc = load_pdf(NEW_PDF_PATH)
    data[NEW_PDF_NAME] = new_doc

    print(f"   → {len(new_doc['words'])} mots | {len(new_doc['lines'])} lignes | {new_doc['n_pages']} pages")

    # Détection sur toutes les pages
    new_results = {}
    for page in range(new_doc['n_pages']):
        tables = detect_tables(new_doc['lines'], page, PARAMS)
        new_results[page] = tables
        if tables:
            print(f"   Page {page+1} → {len(tables)} tableau(x)")
            for k, t in enumerate(tables):
                print(f"      T{k+1}: {t.n_rows}×{t.n_cols} | conf={t.confidence:.0%}")

    results[NEW_PDF_NAME] = new_results

    # Visu page 0
    plot_geometry(new_doc, page=0, title=NEW_PDF_NAME,
                  show_text=True, show_columns=True)
    plot_tables_overlay(new_doc, new_results.get(0, []), page=0,
                        title=NEW_PDF_NAME, show_raw=True, show_cleaned=True)
    display_all_tables(NEW_PDF_NAME, new_results)
else:
    print(f"⚠️  Fichier non trouvé : {NEW_PDF_PATH}")
    print("   → Modifier NEW_PDF_PATH et relancer cette cellule")

---
## Notes & Observations

```
Différences UBS vs Morgan Stanley :

UBS (CH1522817787) :
  - Layout 2 colonnes label/valeur (x0 ≈ 57 / 209 pt)
  - Tableaux intégrés dans le flux (Underlying, Coupon Accrual, Early Redemption)
  - Gaps réguliers → bonne détection par l'algo

Morgan Stanley (BNP Decrement) :
  - Layout tabulaire coloré (x0 ≈ 36 / 183 pt)
  - Grande table autocall sur 3 pages (57 lignes)
  - Gaps très réguliers → détection excellente sur page 2-3
  - Page 1 : table CARACTÉRISTIQUES (2 blocs côte à côte) = cas multi-colonnes

Points d'attention :
  - Les en-têtes de tableaux (gras, 1 seul mot) cassent parfois la régularité
  - max_y_gap doit absorber les séparateurs de sections
  - gap_tolerance = 8 pt est un bon compromis UBS/MS

Prochaines étapes (à intégrer dans le pipeline principal) :
  - Détecter les en-têtes de colonne (ligne précédant le tableau)
  - Gérer les tableaux qui s'étendent sur plusieurs pages
  - Ajouter un score de confiance au niveau champ
```